# ARC-AGI-2 Semantic Quotient Search v1

v0 showed large **syntactic → behavioral redundancy** but zero public-evaluation coverage. v1 changes the experiment:

1. Broaden the generic DSL (geometry, object extraction, bounding boxes, panel logic, inferred scaling, learned recoloring).
2. Prune **during** search rather than only after generation.
3. Use **counterfactual probe grids** in the quotient signature, not demonstrations alone.
4. Keep multiple representatives per quotient class to reduce unsafe over-pruning.
5. Measure **coverage** and **search reduction** separately from final ARC accuracy.

Important: the quotient is still an approximation to semantic equivalence. It means “same behavior on demonstrations + deterministic probes”, not universal equivalence.


In [ ]:
from __future__ import annotations
import json, math, time
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict, Counter
from typing import Callable
import numpy as np, pandas as pd

REQUIRED = {
    "arc-agi_evaluation_challenges.json",
    "arc-agi_evaluation_solutions.json",
    "arc-agi_test_challenges.json",
}
def data_dir():
    roots=[Path("/kaggle/input/competitions"),Path("/kaggle/input"),Path(".")]
    hits=[]
    for root in roots:
        if root.exists():
            for f in root.rglob("arc-agi_test_challenges.json"):
                if REQUIRED.issubset({p.name for p in f.parent.iterdir()}):
                    hits.append(f.parent)
    if not hits:
        raise FileNotFoundError(
            "ARC-AGI-2 competition data is not attached. In Kaggle use Add Input -> "
            "'ARC Prize 2026 - ARC-AGI-2', accept competition rules if requested, then rerun."
        )
    return sorted(hits,key=lambda p:(0 if "competitions" in str(p) else 1,len(str(p))))[0]

D=data_dir()
load=lambda n: json.load(open(D/n))
EV=load("arc-agi_evaluation_challenges.json")
ES=load("arc-agi_evaluation_solutions.json")
TE=load("arc-agi_test_challenges.json")
print("ARC data:",D)
print("evaluation tasks:",len(EV),"visible test tasks:",len(TE))

In [ ]:
def A(g): return np.asarray(g,dtype=np.int8)
def key(x): return tuple(map(tuple,np.asarray(x).tolist()))
def bg(x):
    v,c=np.unique(x,return_counts=True)
    return int(v[np.argmax(c)])

def crop_nonbg(x):
    x=A(x); b=bg(x); pts=np.argwhere(x!=b)
    if not len(pts): return None
    lo,hi=pts.min(0),pts.max(0)
    return x[lo[0]:hi[0]+1,lo[1]:hi[1]+1].copy()

def components(x,diag=False):
    x=A(x); b=bg(x); m=x!=b; H,W=x.shape
    seen=np.zeros_like(m,bool); out=[]
    dirs=[(-1,0),(1,0),(0,-1),(0,1)]
    if diag: dirs += [(-1,-1),(-1,1),(1,-1),(1,1)]
    for r in range(H):
        for c in range(W):
            if not m[r,c] or seen[r,c]: continue
            q=[(r,c)]; seen[r,c]=1; pts=[]
            for rr,cc in q:
                pts.append((rr,cc))
                for dr,dc in dirs:
                    nr,nc=rr+dr,cc+dc
                    if 0<=nr<H and 0<=nc<W and m[nr,nc] and not seen[nr,nc]:
                        seen[nr,nc]=1; q.append((nr,nc))
            out.append(pts)
    return out

def crop_component(x,which):
    x=A(x); cs=components(x)
    if not cs: return None
    comp=max(cs,key=len) if which=="largest" else min(cs,key=len)
    rs=[r for r,c in comp]; csx=[c for r,c in comp]
    r0,r1,c0,c1=min(rs),max(rs),min(csx),max(csx)
    out=np.full((r1-r0+1,c1-c0+1),bg(x),dtype=np.int8)
    for r,c in comp: out[r-r0,c-c0]=x[r,c]
    return out

def fill_bbox(x):
    x=A(x); b=bg(x); pts=np.argwhere(x!=b)
    if not len(pts): return None
    lo,hi=pts.min(0),pts.max(0)
    col=Counter(map(int,x[x!=b])).most_common(1)[0][0]
    y=x.copy(); y[lo[0]:hi[0]+1,lo[1]:hi[1]+1]=col
    return y

def outline_bbox(x):
    x=A(x); b=bg(x); pts=np.argwhere(x!=b)
    if not len(pts): return None
    lo,hi=pts.min(0),pts.max(0)
    col=Counter(map(int,x[x!=b])).most_common(1)[0][0]
    y=np.full_like(x,b); r0,c0=lo; r1,c1=hi
    y[r0,c0:c1+1]=col; y[r1,c0:c1+1]=col
    y[r0:r1+1,c0]=col; y[r0:r1+1,c1]=col
    return y

def split_panels(x):
    x=A(x); H,W=x.shape
    for r in range(1,H-1):
        if np.all(x[r]==x[r,0]):
            a,b=x[:r],x[r+1:]
            if a.shape==b.shape: return a,b
    for c in range(1,W-1):
        if np.all(x[:,c]==x[0,c]):
            a,b=x[:,:c],x[:,c+1:]
            if a.shape==b.shape: return a,b
    return None

def panel_combine(x,op):
    z=split_panels(x)
    if z is None: return None
    a,b=z; ma=a!=bg(a); mb=b!=bg(b)
    if op=="union": m=ma|mb
    elif op=="inter": m=ma&mb
    elif op=="xor": m=ma^mb
    elif op=="a-b": m=ma&~mb
    elif op=="b-a": m=mb&~ma
    else: return None
    return np.where(m,1,0).astype(np.int8)

@dataclass
class P:
    name:str
    cost:float
    fn:Callable
    def __call__(self,g):
        try:
            y=self.fn(A(g))
            if y is None: return None
            y=A(y)
            if y.ndim!=2 or not y.size or max(y.shape)>30: return None
            if y.min()<0 or y.max()>9: return None
            return y
        except Exception:
            return None

def compose(p,q):
    def f(x):
        u=p(x)
        return None if u is None else q(u)
    return P(q.name+"@"+p.name,p.cost+q.cost+.25,f)

def zoom_fn(kr,kc):
    return lambda x:np.repeat(np.repeat(x,kr,0),kc,1)
def tile_fn(kr,kc):
    return lambda x:np.tile(x,(kr,kc))
def down_mode_fn(kr,kc):
    def f(x):
        H,W=x.shape
        if H%kr or W%kc:return None
        y=np.zeros((H//kr,W//kc),dtype=np.int8)
        for r in range(y.shape[0]):
            for c in range(y.shape[1]):
                b=x[r*kr:(r+1)*kr,c*kc:(c+1)*kc].ravel()
                y[r,c]=Counter(map(int,b)).most_common(1)[0][0]
        return y
    return f

def inferred_shape_primitives(task):
    pairs=[]
    for z in task["train"]:
        ih,iw=A(z["input"]).shape; oh,ow=A(z["output"]).shape
        pairs.append((ih,iw,oh,ow))
    ps=[]
    for kr in range(2,5):
        for kc in range(2,5):
            if all(oh==ih*kr and ow==iw*kc for ih,iw,oh,ow in pairs):
                ps += [P(f"zoom{kr}x{kc}",3,zoom_fn(kr,kc)),
                       P(f"tile{kr}x{kc}",3.5,tile_fn(kr,kc))]
            if all(ih==oh*kr and iw==ow*kc for ih,iw,oh,ow in pairs):
                ps += [P(f"downmode{kr}x{kc}",4,down_mode_fn(kr,kc))]
    return ps

def base_programs(task):
    ps=[
      P("id",1,lambda x:x.copy()),
      P("r90",2,lambda x:np.rot90(x,1).copy()),
      P("r180",2,lambda x:np.rot90(x,2).copy()),
      P("r270",2,lambda x:np.rot90(x,3).copy()),
      P("flr",2,lambda x:np.fliplr(x).copy()),
      P("fud",2,lambda x:np.flipud(x).copy()),
      P("T",2,lambda x:x.T.copy()),
      P("crop",2.5,crop_nonbg),
      P("largest",3,lambda x:crop_component(x,"largest")),
      P("smallest",3,lambda x:crop_component(x,"smallest")),
      P("fillbbox",3.5,fill_bbox),
      P("outlinebbox",3.5,outline_bbox),
    ]
    for op in ["union","inter","xor","a-b","b-a"]:
        ps.append(P("panel_"+op,4,lambda x,op=op:panel_combine(x,op)))
    ps += inferred_shape_primitives(task)

    common=set(range(10))
    for z in task["train"]: common &= set(map(int,np.unique(A(z["input"]))))
    for col in sorted(common-{0}):
        def cropc(x,col=col):
            pts=np.argwhere(x==col)
            if not len(pts): return None
            lo,hi=pts.min(0),pts.max(0)
            return x[lo[0]:hi[0]+1,lo[1]:hi[1]+1].copy()
        ps.append(P(f"crop_color_{col}",3,cropc))
    d={}
    for p in ps:
        if p.name not in d or p.cost<d[p.name].cost:d[p.name]=p
    return list(d.values())

def learn_color_map(train,p):
    m={}; changed=False
    for z in train:
        u=p(z["input"]); v=A(z["output"])
        if u is None or u.shape!=v.shape:return None
        for a,b in zip(u.ravel(),v.ravel()):
            a,b=int(a),int(b)
            if a in m and m[a]!=b:return None
            m[a]=b; changed |= a!=b
    if not changed:return None
    def f(x):
        u=p(x)
        if u is None:return None
        y=u.copy()
        for a,b in m.items(): y[u==a]=b
        return y
    return P(p.name+"+map"+str(sorted(m.items())),p.cost+1+.15*len(m),f)

In [ ]:
def synthetic_probes(train):
    shapes=[]
    for z in train:
        s=A(z["input"]).shape
        if s not in shapes: shapes.append(s)
    out=[]
    for H,W in shapes[:3]:
        x=np.zeros((H,W),dtype=np.int8)
        pts=[(0,0,1),(0,W-1,2),(H-1,0,3),(H-1,W-1,4),(H//2,W//3,5)]
        for r,c,v in pts:
            if 0<=r<H and 0<=c<W:x[r,c]=v
        out.append(x)
        y=np.zeros((H,W),dtype=np.int8)
        y[:max(1,H//3),:max(1,W//4)]=2
        if H>2 and W>2:y[-1,-1]=3
        out.append(y)
    return out

def sig(p,inputs):
    return tuple(None if (y:=p(x)) is None else key(y) for x in inputs)

def quotient(programs,inputs,keep=2):
    groups=defaultdict(list)
    for p in programs: groups[sig(p,inputs)].append(p)
    reps=[]
    for g in groups.values():
        reps.extend(sorted(g,key=lambda p:(p.cost,p.name))[:keep])
    return reps,len(groups)

def train_score(p,train):
    exact=0; pixels=[]
    for z in train:
        y=p(z["input"]); t=A(z["output"])
        ok=y is not None and y.shape==t.shape
        exact += int(ok and np.array_equal(y,t))
        pixels.append(float(np.mean(y==t)) if ok else 0.0)
    return exact,float(np.mean(pixels)),p.cost

def build_pool(task,max_depth=3,keep=2,use_probes=True):
    base=base_programs(task)
    demo_inputs=[A(z["input"]) for z in task["train"]]
    probes=synthetic_probes(task["train"]) if use_probes else []
    qinputs=demo_inputs+probes

    attempted=0
    all_reps=[]
    frontier=base
    depth_rows=[]

    for depth in range(1,max_depth+1):
        if depth==1:
            cand=frontier
        else:
            cand=[compose(p,q) for p in frontier for q in base]
        attempted += len(cand)
        reps,nclasses=quotient(cand,qinputs,keep=keep)
        all_reps.extend(reps)
        all_reps,_=quotient(all_reps,qinputs,keep=keep)
        depth_rows.append({
            "depth":depth,"generated":len(cand),
            "probe_classes":nclasses,"kept":len(reps)
        })
        frontier=reps

    mapped=[]
    for p in all_reps:
        m=learn_color_map(task["train"],p)
        if m is not None:mapped.append(m)
    attempted += len(all_reps)
    final,_=quotient(all_reps+mapped,qinputs,keep=keep)

    naive_structural=sum(len(base)**d for d in range(1,max_depth+1))
    return final,{
        "base":len(base),
        "attempted":attempted,
        "naive_structural":naive_structural,
        "search_saving":naive_structural/max(1,attempted),
        "final_reps":len(final),
        "depth_rows":depth_rows,
        "n_probes":len(probes),
    }

def solve(task,max_depth=3,keep=2):
    t0=time.perf_counter()
    cands,diag=build_pool(task,max_depth=max_depth,keep=keep,use_probes=True)
    ranked=sorted(cands,key=lambda p:(
        -train_score(p,task["train"])[0],
        -train_score(p,task["train"])[1],
        p.cost,p.name))
    exact=[p for p in ranked if train_score(p,task["train"])[0]==len(task["train"])]

    answers=[]; ambiguity=[]
    for z in task["test"]:
        source=exact if exact else ranked
        uniq=[]
        for p in source:
            y=p(z["input"])
            if y is None:continue
            ky=key(y)
            if ky not in {u[0] for u in uniq}:uniq.append((ky,y,p.name))
            if len(uniq)>=2:break
        if not uniq:
            y=A(z["input"]);uniq=[(key(y),y,"fallback")]
        if len(uniq)==1:uniq.append(uniq[0])
        answers.append({
            "attempt_1":uniq[0][1].astype(int).tolist(),
            "attempt_2":uniq[1][1].astype(int).tolist()
        })

        if exact:
            pred_classes=set()
            for p in exact:
                y=p(z["input"])
                if y is not None:pred_classes.add(key(y))
            ambiguity.append(len(pred_classes))
        else:
            ambiguity.append(0)

    diag.update({
        "fit_programs":len(exact),
        "fit":int(bool(exact)),
        "test_prediction_classes_max":max(ambiguity) if ambiguity else 0,
        "seconds":time.perf_counter()-t0,
    })
    return answers,diag

In [ ]:
# Public evaluation
MAX_DEPTH=3
KEEP_PER_CLASS=2
LIMIT=None   # e.g. 20 for a quick smoke test

rows=[]; correct=total=0
for i,(tid,task) in enumerate(EV.items()):
    if LIMIT is not None and i>=LIMIT:break
    pred,d=solve(task,MAX_DEPTH,KEEP_PER_CLASS)
    truth=ES[tid]; hit=0
    for p,y in zip(pred,truth):
        y=A(y); a=A(p["attempt_1"]); b=A(p["attempt_2"])
        ok=(a.shape==y.shape and np.array_equal(a,y)) or (b.shape==y.shape and np.array_equal(b,y))
        correct+=int(ok); total+=1; hit+=int(ok)
    rows.append({"task":tid,"correct":hit,"outputs":len(truth),**d})

df=pd.DataFrame(rows)
print("exact output accuracy:",correct/max(1,total))
print("task coverage (>=1 exact demo-fitting program):",df["fit"].mean())
print("median search saving estimate:",df["search_saving"].median())
print("mean search saving estimate:",df["search_saving"].mean())
display(df.head())
display(df[["correct","outputs","base","attempted","naive_structural","search_saving",
            "final_reps","fit_programs","fit","test_prediction_classes_max","seconds"]].describe())
df.to_csv("semantic_quotient_v1_eval.csv",index=False)

In [ ]:
# Hidden-test submission
submission={}; test_rows=[]
for tid,task in TE.items():
    pred,d=solve(task,MAX_DEPTH,KEEP_PER_CLASS)
    submission[tid]=pred
    test_rows.append({"task":tid,**d})

assert set(submission)==set(TE)
for tid,v in submission.items():
    assert len(v)==len(TE[tid]["test"])
    assert all(set(x)=={"attempt_1","attempt_2"} for x in v)

json.dump(submission,open("submission.json","w"))
pd.DataFrame(test_rows).to_csv("semantic_quotient_v1_test.csv",index=False)
print("wrote submission.json for",len(submission),"tasks")

## How to read v1

The most important v1 number is **coverage**, not leaderboard accuracy yet.

- `fit=1`: at least one retained program exactly explains every demonstration.
- `search_saving`: syntactic depth-limited search-tree size divided by candidates actually attempted after quotient pruning. It is an estimate of avoided expansion work, not a hardware-independent complexity theorem.
- `test_prediction_classes_max`: how many distinct unseen-test predictions are produced by demonstration-consistent retained programs. Values greater than 1 expose underdetermination.
- `KEEP_PER_CLASS=2` deliberately preserves more than one cheap representative when probe signatures collide.

The probes are counterfactual diagnostics. They reduce the specific failure of demo-only equivalence, where every exact-fitting program would be collapsed merely because all exact programs must reproduce the demonstrations. Probe-equivalence is still approximate; v1 must be judged by validation accuracy and coverage, not assumed safe.

Next research step after this run: compare probe sets and representative budgets under fixed compute, then introduce information-gain-per-compute scheduling only after coverage is adequate.
